In [1]:
# ==========================================
# Imports
# ==========================================

import numpy as np
from anova_module import ModelAnalysis

In [2]:
# ==========================================
# Context : Function f and support of X
# ==========================================

# Support of X
def generate_joint_support(N: int) -> np.ndarray:
    """
    Generates the tabular support of size r x 5 for the categorical random vector X.
    
    The vector X = (X1, X2, X3, X4, X5) satisfies the following structural equations:
        - X1, X2, X4: Independent variables with support {0, ..., N-1}.
        - X3: Deterministic variable where X3 = X2 almost surely.
        - X5: Deterministic constant where X5 = 0 almost surely.
    
    The resulting support size is r = N^3.

    Parameters
    ----------
    N : int
        The cardinality of the sample space for the independent variables.

    Returns
    -------
    np.ndarray
        A matrix of shape (N^3, 5) containing all possible realizations of the vector X.
    """
    
    # 1. Generate the grid for the independent variables X1, X2, X4.
    # We generate indices for a 3D grid of shape (N, N, N).
    # Reshape and transpose to obtain the Cartesian product of size (N^3, 3).
    grid = np.indices((N, N, N)).reshape(3, -1).T
    
    # Extract independent components from the grid
    x1 = grid[:, 0]
    x2 = grid[:, 1]
    x4 = grid[:, 2]
    
    # 2. Construct the dependent/deterministic variables
    x3 = x2                     # Constraint: X3 copies X2 (perfect correlation)
    x5 = np.zeros(N**3, dtype=int)  # Constraint: X5 is constant at 0
    
    # 3. Stack all components to form the joint support matrix
    support = np.column_stack((x1, x2, x3, x4, x5))
    
    return support

# Function f
def compute_linear_threshold(X: np.ndarray) -> np.ndarray:
    """
    Computes the sign of a fixed linear combination of the first three variables.
    
    Given an input matrix X of shape (n, 5), this function calculates:
        y = sign(a * X1 + b * X2 + c * X3)
        
    where a, b, and c are fixed hyperparameters defined within the function.
    
    Parameters
    ----------
    X : np.ndarray
        Input data matrix of shape (n, 5).
        Columns must correspond to [X1, X2, X3, X4, X5].

    Returns
    -------
    np.ndarray
        Output vector of shape (n,) containing values {-1, 0, 1}.
    """
    # ---------------------------------------------------------
    # Fixed Hyperparameters (Hardcoded as requested)
    # ---------------------------------------------------------
    ALPHA = 1   # Coef for X1
    BETA  = -1  # Coef for X2
    GAMMA = 0.5   # Coef for X3
    
    # ---------------------------------------------------------
    # Vectorized Computation
    # ---------------------------------------------------------
    # We use column slicing X[:, i] to perform operations on the entire 
    # dataset at once. This leverages BLAS optimization under the hood.
    
    linear_combination = (
        ALPHA * X[:, 0] + 
        BETA  * X[:, 1] + 
        GAMMA * X[:, 2]
    )
    
    # np.sign returns -1 if x < 0, 0 if x == 0, 1 if x > 0
    return np.sign(linear_combination)

In [3]:
# ==========================================
# Parameters and application of our framework
# ==========================================

d = 5 # number of variables
N = 3 # number of modalities for each rv
X = generate_joint_support(N) # support
f_model = compute_linear_threshold # function
A = ModelAnalysis(X , f_model , 100 , 1e-3 , 1e-10) # class
S , Matrix = A.functional_anova() # sets and f_A
P = A.get_P() # probabilities

Constructing Basis Matrix:   0%|          | 0/27 [00:00<?, ?it/s]

Constructing Basis Matrix: 100%|██████████| 27/27 [00:00<00:00, 4777.11it/s]

Computations complete. Results ready.


In [4]:
# Sets S

S

[[], [1], [2], [4], [1, 2], [1, 4], [2, 4], [1, 2, 4]]

In [5]:
# Norms of f_S(X_S) (in L^2 equipped with its scalar product)

np.sum( (Matrix**2).T * P , axis=1 )

array([1.11111111e-01, 5.18518518e-01, 7.40740741e-02, 9.39334830e-37,
       7.40740741e-02, 8.34964294e-37, 8.34964294e-37, 0.00000000e+00])